### Step 1: Import API keys and libraries

In [2]:
import os
import json
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API_KEY is missing.")

/workspaces/AIchallenge/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 2: Simple RAG

In [1]:
from document_chunker import chunk_document

with open("document.txt") as f:
    text = f.read()

chunks = chunk_document(text, chunk_size=800, chunk_overlap=100,
                        metadata={"source": "your_document.txt"})

print(f"{len(chunks)} chunks")
chunks[0]  # inspect the first one

47 chunks


Chunk(text='Netflix Culture\n\nEntertainment, like friendship, is a fundamental human need; it changes how we feel and gives us common ground. We want to entertain the world. If we succeed, there is more laughter, more empathy, and more joy. \n\nTo get there, we have an amazing and unusual employee culture. This document is about that culture. \n\nLike all great companies, we strive to hire the best and we value integrity, excellence, respect, inclusion, and collaboration. What is special about Netflix, though, is how much we:\n\n', index=0, start_char=0, end_char=516, metadata={'source': 'your_document.txt', 'chunk_index': 0})

In [3]:
def display_chunks_md(chunks, n=None):
    """Render chunks as markdown. If n is given, only show the first n."""
    subset = chunks[:n] if n is not None else chunks
    for c in subset:
        md = (
            f"### Chunk {c.index}  \n"
            f"*chars {c.start_char}–{c.end_char} · ~{c.token_estimate} tokens*  \n"
            f"*metadata:* `{c.metadata}`\n\n"
            f"> {c.text.strip().replace(chr(10), chr(10) + '> ')}\n\n"
            f"---"
        )
        display(Markdown(md))
    if n is not None and n < len(chunks):
        display(Markdown(f"*...{len(chunks) - n} more chunks not shown*"))

display_chunks_md(chunks, n=5)

### Chunk 0  
*chars 0–516 · ~129 tokens*  
*metadata:* `{'source': 'your_document.txt', 'chunk_index': 0}`

> Netflix Culture
> 
> Entertainment, like friendship, is a fundamental human need; it changes how we feel and gives us common ground. We want to entertain the world. If we succeed, there is more laughter, more empathy, and more joy. 
> 
> To get there, we have an amazing and unusual employee culture. This document is about that culture. 
> 
> Like all great companies, we strive to hire the best and we value integrity, excellence, respect, inclusion, and collaboration. What is special about Netflix, though, is how much we:

---

### Chunk 1  
*chars 416–946 · ~132 tokens*  
*metadata:* `{'source': 'your_document.txt', 'chunk_index': 1}`

> nce, respect, inclusion, and collaboration. What is special about Netflix, though, is how much we:
> 
> encourage independent decision-making by employees
> share information openly, broadly, and deliberately
> are extraordinarily candid with each other
> keep only our highly effective people
> avoid rules
> Our core philosophy is people over process. More specifically, we have great people working together as a dream team. With this approach, we are a more flexible, fun, stimulating, creative, collaborative and successful organization.

---

### Chunk 2  
*chars 846–1641 · ~198 tokens*  
*metadata:* `{'source': 'your_document.txt', 'chunk_index': 2}`

> ch, we are a more flexible, fun, stimulating, creative, collaborative and successful organization.
> 
> Real Values
> Many companies have value statements, but often these written values are vague and ignored. The real values of a firm are shown by who gets rewarded or let go. Below are our values, the specific behaviors and skills we care about most. The more these values sound like you, and describe people you want to work with, the more likely you will thrive at Netflix. 
> 
> Judgment
> You make wise decisions despite ambiguity 
> You identify root causes, and get beyond treating symptoms 
> You think strategically, and can articulate what you are, and are not, trying to do 
> You are good at using data to inform your intuition
> You make decisions based on the long term, not near term
> Communication

---

### Chunk 3  
*chars 1541–2265 · ~181 tokens*  
*metadata:* `{'source': 'your_document.txt', 'chunk_index': 3}`

> ata to inform your intuition
> You make decisions based on the long term, not near term
> Communication
> You are concise and articulate in speech and writing 
> You listen well and seek to understand before reacting 
> You maintain calm poise in stressful situations to draw out the clearest thinking
> You adapt your communication style to work well with people from around the world who may not share your native language
> You provide candid, helpful, timely feedback to colleagues
> Curiosity
> You learn rapidly and eagerly 
> You contribute effectively outside of your specialty 
> You make connections that others miss
> You seek to understand our members around the world, and how we entertain them
> You seek alternate perspectives
> Courage

---

### Chunk 4  
*chars 2165–2937 · ~193 tokens*  
*metadata:* `{'source': 'your_document.txt', 'chunk_index': 4}`

> and our members around the world, and how we entertain them
> You seek alternate perspectives
> Courage
> You say what you think, when it’s in the best interest of Netflix, even if it is uncomfortable 
> You make tough decisions without agonizing 
> You take smart risks and are open to possible failure
> You question actions inconsistent with our values 
> You are able to be vulnerable, in search of truth
> Passion
> You inspire others with your thirst for excellence 
> You care intensely about our members and Netflix‘s success 
> You are tenacious and optimistic
> You are quietly confident and openly humble
> Selflessness
> You seek what is best for Netflix, rather than what is best for yourself or your group 
> You are open-minded in search of great ideas
> You make time to help colleagues

---

*...42 more chunks not shown*

### Step 3: Chunking

### Step 4: Create notification function

In [4]:
tools = []

In [5]:
pushover_user = os.getenv("PUSHOEVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

import requests 
def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)
    
send_notification_function = {
    "name": "send_notification",
    "description": "Send a notification to the real-world version of you via Pushover.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send as a notification."
            }
        },
        "required": ["message"]
    }
}

tools.append({"type": "function", "function":send_notification_function})

In [6]:
import random

def dice_roll():
    result = random.randint(1, 6)
    return result

roll_dice_function = {
    "name": "dice_roll",
    "description": "Simulates rolling a single six-sided die and returns the result.  Use this when the user wants to roll a die for games, dcisions, or randcom numbers.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

tools.append({"type": "function", "function": roll_dice_function})

### Step 5: Define tools and functions for the agent to use

In [10]:
def handle_tool_call(tool_calls):
    tool_results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        
        if function_name == "send_notification":
            send_notification(args["message"])
            content = f"Notification sent with message: {args['message']}"
        elif function_name == "dice_roll":
            content = f"Die rolled and the result is: {dice_roll()}"
        else:
            content = f"Unknown tool: {function_name}"
        
        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }
        
        tool_results.append(tool_call_result)
        
    return tool_results

### Step 6: Initiate chatbot loop

In [11]:
def respond_ai(message, history):
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        keyword = keyword.split()
        for k in keyword:
            if k in message.lower():
                system_message_enhanced += f"\n\n{context}"
    
    messages = [{"role": "system", "content": system_message_enhanced + "***"}]  + history + [{"role": "user", "content": message}]    
    
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )
    
    message = response.choices[0].message
    
    while message.tool_calls:
        from pprint import pprint
        pprint(message.tool_calls)
            
        tool_result = handle_tool_call(message.tool_calls)
        messages.append(message)
        messages.extend(tool_result)

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools
        )
        
        message = response.choices[0].message
    
    return(message.content)

    


In [12]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True, inline=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


[ChatCompletionMessageFunctionToolCall(id='call_lvRKolWWxn6scdej6v0Y43uM', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_dD5HrSqM4hiE6VvGWHgwwFDj', function=Function(arguments='{}', name='dice_roll'), type='function')]
[ChatCompletionMessageFunctionToolCall(id='call_UJTKd0fNxNg75NI8AFdq9s0i', function=Function(arguments='{"message":"The highest roll value from the first two dice is 5."}', name='send_notification'), type='function')]
[ChatCompletionMessageFunctionToolCall(id='call_3gcL6dkqOqce5zLt9dV3P3bm', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_vajawC0lSrS2KGCkXBEtewUZ', function=Function(arguments='{}', name='dice_roll'), type='function')]
[ChatCompletionMessageFunctionToolCall(id='call_348wtNvK3kfM4ZSrjLimi1CL', function=Function(arguments='{"message":"The highest roll value from all four dice rolls is 5."}', name='send_notification'